# Example use-case: LISA UCB convolution of double white-dwarf binaries at current-day

This example fleshes out the steps required to estimate the population of observable double white dwarf systems in the LISA band. Relevant studies are: https://arxiv.org/abs/2405.20484

Several ingredients are necessary here:
- population-synthesis results that contain white-dwarfs
- a Milky-Way galaxy star formation rate history model
- a method to evolve 

Convolution-by-sampling was developed especially for this project, as we want to 'generate' double white dwarf systems at a certain lookback time, and evolve them (through gravitational-wave radiation) to the present day.

The convolution broadly is done as follows:
- In a given lookback-time bin we calculate the total mass formed into stars
- We use that to generate double white dwarf systems (using mass_formed * yield-per-mass-formed) 
- We assign a birth time to these systems (with values bound by the edges of the lookback time bin)
- We 'evolve' these systems up to the current day under the influence of gravitational-wave radiation. We make use of Legwork ([Wagg et al 2021](https://ui.adsabs.harvard.edu/abs/2022ApJS..260...52W/abstract)) in this example.
- Filter out certain systems (in particular those that, at current-day, are not in the LISA waveband)
- Calculate detection probabilities for the rest based on their position (either randomly assigned or motivated by a spatially-defined SFH) in the Milkyway and their system properties.
- Use this information to predict observable populations of DWD systems

In the following notebook I will show how to set up all the necessary pieces for this script and how to put them together. We will start some general imports like usual, then continue with setting up the `sfr_dict`, then design the post convolution function, and then combine everything and run the convolution.

Lets start with setting up some form of star formation rate for the milkyway. There are many estimates and descriptions of increasing complexity.

In [1]:
"""
Functions to convolve the T0 format with sampling
"""

import os
import json
import time
import copy
import astropy.units as u
import legwork as lw
import numpy as np
import astropy.constants as const
import pandas as pd
import pkg_resources
import h5py

from syntheticstellarpopconvolve import convolve, default_convolution_config
from syntheticstellarpopconvolve.general_functions import temp_dir
from syntheticstellarpopconvolve.usecase_notebook_utils.usecase_lisa_utils import get_mass_norm, sample_distances_simple, get_period
from syntheticstellarpopconvolve.convolve_stochastically import (
    select_dict_entries_with_new_indices,
)

TMP_DIR = temp_dir("code", "convolve_stochastically", clean_path=True)

# The flag below allows the user to run this notebook without the full data or starformation rate. 
FULL_VERSION = os.getenv("EXAMPLE_USECASE_UCB_VERSION")

Next step is to load some data. In this case we have loaded some data expressed in the T0 bincodex format, and sourced from SeBa simulations, which are Monte-Carlo based (opposed to grid-based) simulations. We want to select double white-dwarf systems only, and provide a `normalized_yield` to them.

In [2]:
data_filename = os.getenv('EXAMPLE_DATA_USECASE_UCB_FILENAME') if FULL_VERSION else None
example_usecase_UCB_events_filename = data_filename if data_filename is not None else pkg_resources.resource_filename(
    "syntheticstellarpopconvolve",
    "example_data/example_BinCodex_dwd.h5"
)

#
example_usecase_UCB_events_filename="/home/david/Desktop/data_bincodex/bincodex_results/Seba_BinCodex.h5"

example_usecase_UCB_events_data = pd.read_hdf(example_usecase_UCB_events_filename,  key='T0')

# BinCodex_events_filename = (
#     "/home/david/Desktop/bincodex_results/example_BinCodex.h5"
# )

##################
# Provide the normalized_yield and select only systems that are double white-dwards

# get mass normalisation
mass_normalisation_fiducial = get_mass_norm(IC_model="fiducial", binary_fraction=0.5)

# set normalised yield
example_usecase_UCB_events_data["normalized_yield"] = 1 / mass_normalisation_fiducial
# Query the dataset to select the formation of the WDs

# check if things start with some number. It's easier to turn them into strings for this.
example_usecase_UCB_events_data["str_event"] = example_usecase_UCB_events_data["event"].astype(str)
example_usecase_UCB_events_data["str_type1"] = example_usecase_UCB_events_data["type1"].astype(str)
example_usecase_UCB_events_data["str_type2"] = example_usecase_UCB_events_data["type2"].astype(str)

# lets query the type-changing events. Any type-change will do
wd_binaries = example_usecase_UCB_events_data.query("str_event.str.startswith('1')")

# The type should change to a WD-type (and the other should already be one)
wd_binaries = wd_binaries.query("str_type1.str.startswith('2')")
wd_binaries = wd_binaries.query("str_type2.str.startswith('2')")

# Lets delete the string versions of the columns again
wd_binaries = wd_binaries.drop(columns=["str_event", "str_type1", "str_type2"])

# lets also delete the original dataframe
del example_usecase_UCB_events_data

Lets now set up some form of star formation rate for the Milkyway. There are many estimates and descriptions of increasing complexity, but for simplicity lets take a constant star formation that within 10Gyr will have formed a mass equivalent to the stellar mass of the Milky way. We will ignore metallicity here.

In [3]:
sfr_dict = {}
scale = 1e-2

# Set up the lookback time bins
if FULL_VERSION:
    sfr_dict["lookback_time_bin_edges"] = (np.arange(0, 10, 0.25) * u.Gyr).to(u.yr)
else:
    sfr_dict["lookback_time_bin_edges"] = (np.arange(0, 10, 0.1) * u.Gyr).to(u.yr)

#
sfr_dict["starformation_rate_array"] = (6 * u.Msun / u.yr) *  np.ones(sfr_dict["lookback_time_bin_edges"].shape[0] - 1)

if not FULL_VERSION:
    sfr_dict["starformation_rate_array"] *= scale

With the starformation information now configured, we can design the actual machinery of this calculation: the post-convolution step.

Convolution by sampling 'generates' systems, and returns indices to those systems that allows us to link back to system properties. It also provides formation times that we can use.

The idea in this routine will be to:
- Use the 'generated' system (indices)
- ~Calculate the the time difference between current day (lookback time = 0) and the lookback time where the event occurred (lookback time = formation time + delay time).~ This is done automatically.
- Use this time to 'evolve' the double white dwarf system forward in time under the influence of gravitational wave radiation. (we use Legwork).
- Filter out systems that fall outside of the LISA waveband.

This is what we will handle within the post-convolution. We will thus store every system that falls within the LISA waveband at current day. We will handle the processing of detectability in the next step. 

In [4]:
def post_convolution_function(
    config, sfr_dict, data_dict, convolution_results, convolution_instruction
):
    """
    Post-convolution function to handle integrating the systems forward in time and finding those that end up in the LISA waveband.

    using local_indices to select everything and using Alexey's distance sampler to handle sampling the distances
    """

    # unpack data
    system_indices = convolution_results["indices"] # These allow linking back to the data of the systems
    event_lookback_times = convolution_results["event_lookback_times"] # These are the lookback times when the events occur. 
                                                               # 'formation_lookback_times' are also available.
    local_indices = np.arange(len(system_indices))

    # select system properties using the indices
    sma = data_dict["semimajor_axis"][system_indices] * u.Rsun
    m_1 = data_dict["mass1"][system_indices] * u.Msun
    m_2 = data_dict["mass2"][system_indices] * u.Msun 
    eccentricity = data_dict["eccentricity"][system_indices]
    periods = get_period(sma, m_1, m_2)
    f_orb_i = (1 / periods).to(u.Hz)  

    # sample distances and add those to the result dict
    dist = sample_distances_simple(NBin=len(system_indices))
    convolution_results["dists"] = dist
    
    #########
    # Set up sources In Legwork
    sources = lw.source.Source(
        m_1=m_1,
        m_2=m_2,
        ecc=eccentricity,
        f_orb=f_orb_i,
        dist=dist,
        interpolate_g=len(local_indices) > 1000,
    )

    #########
    # Evolve the systems until today
    t_evol = event_lookback_times
    sources.evolve_sources(t_evol)
    f_orb_now = sources.f_orb # Get the orbital frequencies of the systems at current-day.
    convolution_results["f_orb_now"] = f_orb_now # Store the orbital frequency in the result dict

    ####
    # categorisations
    lower_bound_LISA_passband = 1e-5 * u.Hz
    upper_bound_LISA_passband = 1e-1 * u.Hz

    # 1) doesnt enter lisa waveband today. so also nt in the past (maybe near future)
    # 2) are currently in lisa band. maybe also in the past (but fro which point)
    # 3) are merged now. but they ahve been in lisa band in the past (and from which point)
    # 4) for both 2 and 3, we should filter out the 'interacting' systems

    ##############
    # determine (un)merged systems 
    # Whether a system is merged is determined checking if its orbital frequency is above 100hz
    unmerged_mask = f_orb_now < 1e2 * u.Hz
    config["logger"].warning(
        f"Of the total of {len(local_indices)} systems {np.count_nonzero(unmerged_mask==0)} are merged by today and {np.count_nonzero(unmerged_mask)} are not"
    )

    # f_orb_now_unmerged_systems = f_orb_now[f_orb_now < 1e2 * u.Hz]

    ##############
    # determine unmerged systems in LISA passband
    waveband_mask = (f_orb_now >= lower_bound_LISA_passband) & (f_orb_now <= upper_bound_LISA_passband)

    config["logger"].warning(
        f"Of the total of {len(local_indices)} systems {np.count_nonzero(waveband_mask)} are within the lisa frequency passband ([{lower_bound_LISA_passband},{upper_bound_LISA_passband}])"
    )

    #######
    # filter based on detectability here
    snr = sources.get_snr(verbose=True)
    detectability_mask = snr > 7

    config["logger"].warning(
        f"Of the total of {len(local_indices)} systems {np.count_nonzero(detectability_mask)} have a signal to noise (SNR) ratio of above 7 in an observation period of 4 years"
    )

    ##############
    # combine masks and construct filtered index list
    combined_mask = unmerged_mask * waveband_mask * detectability_mask

    masked_local_indices = local_indices[combined_mask]

    #######
    # return only data from now unmerged systems within the lisa passband
    # We use the function `select_dict_entries_with_new_indices` to select the data using the new indices in every entry in the dict. 
    # Result dict does not contain that many 
    convolution_results = select_dict_entries_with_new_indices(
        sampled_data_dict=convolution_results,
        new_indices=masked_local_indices,
    )
    
    return convolution_results

Lastly, we set up the `convolution_config`, which binds everything together. Most of this is similar to other scripts, but here we use a different kind of convolution, namely convolution by sampling. Information about this type of sampling is available **here TODO: link**

In [5]:
##################
#

# create file
input_hdf5_filename = os.path.join(TMP_DIR, "input_hdf5.h5")
output_hdf5_filename = os.path.join(TMP_DIR, "output_hdf5.h5")
input_hdf5_file = h5py.File(input_hdf5_filename, "w")

# Create groups main
input_hdf5_file.create_group("input_data")
input_hdf5_file.create_group("config")

# add group for events
input_hdf5_file.create_group("input_data/events")

# # Write population config to file
# input_hdf5_file.create_dataset("config/population", data=json.dumps({}))

# close
input_hdf5_file.close()

# store the data frame in the hdf5file
wd_binaries.to_hdf(input_hdf5_filename, key="input_data/events/example_usecase_UCB_events")

#
convolution_config = copy.copy(default_convolution_config)
convolution_config["input_filename"] = input_hdf5_filename
convolution_config["output_filename"] = output_hdf5_filename
convolution_config["tmp_dir"] = TMP_DIR
convolution_config["redshift_interpolator_data_output_filename"] = os.path.join(
    TMP_DIR, "interpolator_dict.p"
)

###
# convolution instructions
convolution_config["convolution_instructions"] = [
    {
        "input_data_type": "event",
        "convolution_type": "sample",
        "input_data_name": "example_usecase_UCB_events",
        "output_data_name": "example_usecase_UCB_events",
        "ignore_metallicity": True,
        "post_convolution_function": post_convolution_function,
        "data_column_dict": {
            # required
            "normalized_yield": "normalized_yield",
            "delay_time": {"column_name": "time", "unit": u.Myr},
            # The columns below are required because we use them in the post_convolution function. We can either give their units here, or we can assign them in the post-convolution function.
            "semimajor_axis": "semiMajor",
            "mass1": "mass1",
            "mass2": "mass2",
            "eccentricity": "eccentricity"
        },
    },
]

# Configure time type
convolution_config["time_type"] = "lookback_time"

# store SFR dict
convolution_config["SFR_info"] = sfr_dict

In [6]:
# convolve
convolve(config=convolution_config)

print("finished convolution")

[convolve_stochastically.py:234 - calculate_total_star_formation_in_bin ] 2025-01-22 09:53:43,139: Lower time bin 0.0 yr upper time bin 2000000000.0 yr total mass formed 120000000.0 solMass
[convolve_stochastically.py:289 -       sample_systems ] 2025-01-22 09:53:43,146: Convolving through sampling. Using a total of 120000000.0 solMass
[convolve_stochastically.py:351 -       sample_systems ] 2025-01-22 09:53:43,282: Sampled 2688051 systems.
[convolve_stochastically.py:152 - add_event_lookback_time_and_filter ] 2025-01-22 09:53:43,284: Adding event lookback time.
[convolve_stochastically.py:183 - add_event_lookback_time_and_filter ] 2025-01-22 09:53:43,336: Filtering out 1737077 systems that would occur in the future. 950974 systems are left, and happen in the past
[convolve_stochastically.py:76 - convolve_events_by_sampling_post_convolution_hook_wrapper ] 2025-01-22 09:53:43,378: Handling post-convolution function hook call for convolve-events by sampling
/home/david/.pyenv/versions/3.

Calculating SNR for 950974 sources
	41011 sources have already merged
	909963 sources are stationary
		909963 sources are stationary and circular


[4271153920.py:79 - post_convolution_function ] 2025-01-22 09:55:13,732: Of the total of 950974 systems 65 have a signal to noise (SNR) ratio of above 7 in an observation period of 4 years
[convolve_stochastically.py:234 - calculate_total_star_formation_in_bin ] 2025-01-22 09:55:13,771: Lower time bin 2000000000.0 yr upper time bin 4000000000.0 yr total mass formed 120000000.0 solMass
[convolve_stochastically.py:289 -       sample_systems ] 2025-01-22 09:55:13,774: Convolving through sampling. Using a total of 120000000.0 solMass
[convolve_stochastically.py:351 -       sample_systems ] 2025-01-22 09:55:14,014: Sampled 2688332 systems.
[convolve_stochastically.py:152 - add_event_lookback_time_and_filter ] 2025-01-22 09:55:14,017: Adding event lookback time.
[convolve_stochastically.py:183 - add_event_lookback_time_and_filter ] 2025-01-22 09:55:14,126: Filtering out 974170 systems that would occur in the future. 1714162 systems are left, and happen in the past
[convolve_stochastically.py

Calculating SNR for 1714162 sources
	98152 sources have already merged
	1616010 sources are stationary
		1616010 sources are stationary and circular


[4271153920.py:79 - post_convolution_function ] 2025-01-22 09:58:35,782: Of the total of 1714162 systems 49 have a signal to noise (SNR) ratio of above 7 in an observation period of 4 years
[convolve_stochastically.py:234 - calculate_total_star_formation_in_bin ] 2025-01-22 09:58:35,823: Lower time bin 4000000000.0 yr upper time bin 6000000000.0 yr total mass formed 120000000.0 solMass
[convolve_stochastically.py:289 -       sample_systems ] 2025-01-22 09:58:35,825: Convolving through sampling. Using a total of 120000000.0 solMass
[convolve_stochastically.py:351 -       sample_systems ] 2025-01-22 09:58:35,963: Sampled 2687994 systems.
[convolve_stochastically.py:152 - add_event_lookback_time_and_filter ] 2025-01-22 09:58:35,964: Adding event lookback time.
[convolve_stochastically.py:183 - add_event_lookback_time_and_filter ] 2025-01-22 09:58:36,046: Filtering out 658068 systems that would occur in the future. 2029926 systems are left, and happen in the past
[convolve_stochastically.p

Calculating SNR for 2029926 sources
	116777 sources have already merged
	1913149 sources are stationary
		1913149 sources are stationary and circular


[4271153920.py:79 - post_convolution_function ] 2025-01-22 10:01:47,936: Of the total of 2029926 systems 5 have a signal to noise (SNR) ratio of above 7 in an observation period of 4 years
[convolve_stochastically.py:234 - calculate_total_star_formation_in_bin ] 2025-01-22 10:01:47,977: Lower time bin 6000000000.0 yr upper time bin 8000000000.0 yr total mass formed 120000000.0 solMass
[convolve_stochastically.py:289 -       sample_systems ] 2025-01-22 10:01:47,980: Convolving through sampling. Using a total of 120000000.0 solMass
[convolve_stochastically.py:351 -       sample_systems ] 2025-01-22 10:01:48,109: Sampled 2688075 systems.
[convolve_stochastically.py:152 - add_event_lookback_time_and_filter ] 2025-01-22 10:01:48,111: Adding event lookback time.
[convolve_stochastically.py:183 - add_event_lookback_time_and_filter ] 2025-01-22 10:01:48,175: Filtering out 448551 systems that would occur in the future. 2239524 systems are left, and happen in the past
[convolve_stochastically.py

Calculating SNR for 2239524 sources
	120368 sources have already merged
	2119156 sources are stationary
		2119156 sources are stationary and circular


[4271153920.py:79 - post_convolution_function ] 2025-01-22 10:05:23,196: Of the total of 2239524 systems 0 have a signal to noise (SNR) ratio of above 7 in an observation period of 4 years
[convolve_populations.py:113 - store_convolution_result_entries ] 2025-01-22 10:05:23,276: Storing delay_time
[convolve_populations.py:113 - store_convolution_result_entries ] 2025-01-22 10:05:23,280: Storing semimajor_axis
[convolve_populations.py:113 - store_convolution_result_entries ] 2025-01-22 10:05:23,282: Storing mass1
[convolve_populations.py:113 - store_convolution_result_entries ] 2025-01-22 10:05:23,285: Storing mass2
[convolve_populations.py:113 - store_convolution_result_entries ] 2025-01-22 10:05:23,290: Storing eccentricity
[convolve_populations.py:113 - store_convolution_result_entries ] 2025-01-22 10:05:23,294: Storing indices
[convolve_populations.py:113 - store_convolution_result_entries ] 2025-01-22 10:05:23,296: Storing formation_lookback_times
[convolve_populations.py:113 - sto

finished convolution


Now that the convolution is finished, we can read out the results. Remember, in this case, the bins in which the data is stored indicate the range of lookback times in which the systems are born. They are all evolved to the current day, however, so we need to add the results of all these bins together to get a complete view of the population of dwd systems that will be observable by LISA!

In [11]:


# read out content and integrate until today
with h5py.File(convolution_config["output_filename"], "r") as output_hdf5_file:
    group_key = "output_data/event/example_usecase_UCB_events/example_usecase_UCB_events/convolution_results"
    
    formation_time_bin_keys = list(
        output_hdf5_file[group_key].keys()
    )

    # ################
    # #
    # total_in_waveband_lisa = 0

    # loop over the formation-time bins
    formation_time_bin_keys = sorted(
        formation_time_bin_keys, key=lambda x: float(x.split(" ")[0])
    )
    for formation_time_bin_key in formation_time_bin_keys:
        # 
        time_bin_group_key = f"{group_key}/{formation_time_bin_key}"
            
        # formation_time_bin_key = "3500000000.0 yr"
        print("=================================")
        print(f"formation_time_bin_key: {formation_time_bin_key}")
        print("=================================")

        
        #####
        # convert units
        unit_dict = json.loads(
            output_hdf5_file[
                f"{group_key}/{formation_time_bin_key}"
            ].attrs["units"]
        )
        unit_dict = {key: u.Unit(val) for key, val in unit_dict.items()}

        print(unit_dict)

        ###########
        # Read out data
        convolution_results = output_hdf5_file[time_bin_group_key].keys()
        


        eccentricity = output_hdf5_file[f'{time_bin_group_key}/eccentricity'][()]
        f_orb_now = output_hdf5_file[f'{time_bin_group_key}/f_orb_now'][()] * unit_dict['f_orb_now']     
        mass1 = output_hdf5_file[f'{time_bin_group_key}/mass1'][()] * u.Msun
        mass2 = output_hdf5_file[f'{time_bin_group_key}/mass2'][()] * u.Msun
        dists = output_hdf5_file[f'{time_bin_group_key}/dists'][()] * unit_dict['dists']
        print(f_orb_now)
        # #########
        # # TODO: this can be done in the post-convolution step itself
        # # Set up sources In Legwork
        sources = lw.source.Source(
            m_1=mass1,
            m_2=mass2,
            ecc=eccentricity,
            f_orb=f_orb_now,
            dist=dists,
            interpolate_g=True,
        )

        # calculate the SNR
        snr = sources.get_snr(verbose=True)
        print(snr)
        
        # # mask detectable binaries
        # detectable = snr > 7
        
        # print(detectable)

        

formation_time_bin_key: 1000000000.0 yr
{'delay_time': Unit("Myr"), 'formation_lookback_times': Unit("yr"), 'event_lookback_times': Unit("yr"), 'dists': Unit("kpc"), 'f_orb_now': Unit("Hz")}
[0.00256132 0.0013294  0.00127488 0.00150629 0.00183694 0.00149658
 0.00173815 0.00246152 0.00155224 0.00193229 0.00204863 0.00122065
 0.0014145  0.00113807 0.00132346 0.0019952  0.00102929 0.00098953
 0.00099216 0.00214541 0.00109974 0.0020272  0.00137185 0.00070987
 0.00075972 0.001095   0.00141418 0.00100182 0.00080831 0.0009712
 0.00065019 0.00047275 0.00115179 0.00346077 0.00168368 0.00135407
 0.00125963 0.00052631 0.00086739 0.0010167  0.00171417 0.00171076
 0.00077098 0.00062957 0.00071023 0.00083074 0.00092309 0.00107494
 0.00096193 0.00186092 0.00150464 0.00112487 0.00170509 0.00038917
 0.00080433 0.00113434 0.00122157 0.00034034 0.00092782 0.00098547
 0.00265632 0.00146847 0.00070393 0.00069294 0.00051816] Hz
Calculating SNR for 65 sources
	0 sources have already merged
	65 sources are st

In [6]:
import astropy.units as u
import legwork as lw

# -----------------------------
# 1) Define your source parameters
# -----------------------------
# For a double white-dwarf binary:
m_1 = 0.3 * u.Msun   # mass of the first WD
m_2 = 0.3 * u.Msun   # mass of the second WD
eccentricity = 0

# Orbital frequency (f_orb): frequency of the binary's orbit
# Note that the gravitational wave frequency for a circular orbit is 2*f_orb
f_orb = 1.0e-3 * u.Hz

# Distance to the source
dist = 1000.0 * u.pc

# -----------------------------
# 2) Create a Source object
# -----------------------------
# The Source class in LEGWORK can take arrays or scalars of parameters. 
# Here we demonstrate with single (scalar) values.
binary = lw.source.Source(
    m_1=m_1,
    m_2=m_2,
    ecc=eccentricity,
    f_orb=f_orb,
    dist=dist,
    # interpolate_g=len(local_indices) > 1000,
)

# -----------------------------
# 3) Compute the SNR
# -----------------------------
# By default, LEGWORK uses the standard LISA sensitivity PSD from Robson et al. (2019).
snr = binary.get_snr()
print(snr)
# print(f"SNR = {snr:.3f}")

# -----------------------------
# 4) Compute the detection probability
# -----------------------------
# LEGWORK uses a “characteristic SNR threshold” approach by default 
# (commonly ~7 for a "resolvable" detection in many LISA studies).
# You can override the default threshold via the "snr_thresh" argument if desired.
# p_det = binary.get_detection_probability()
# print(p_det)
# print(p_det
# # p_det = binary.get_detection_probability()


# Check if the binary is detectable above SNR=7
is_detectable = binary.detectable(snr_threshold=7.0)
print(is_detectable)  # returns True or False
      
# print(f"Detection probability = {p_det:.3f}")


[46.94633165]


AttributeError: 'Source' object has no attribute 'detectable'

## Advanced steps
This example is a good step towards a solid predictive calculation for the population of observable white dwarf systems for the LISA mission, but it does lack some sophistication. In particular, the star formation history information is not so realistic. 

This example can be made more sophisticated by e.g.:
- Using a spatially-defined star-formation rate history. One can provide a list of starformation histories to the code, each element then representing a part of the grid where the SFR is defined in.
- Splitting off systems that may undergo RLOF
- providing on-sky angle (dec, asc), inclination of system relative to us, orbital phase for eccentric systems

https://legwork.readthedocs.io/en/latest/notebooks/Source.html#Position-inclination-polarisation-specfic-sources

TODO: determine which systems that are (at present day) in the lisa frequency range should have interacted through RLOF
TODO: of the systems that are not RLOFing and are within the lisa waveband, store: indices, source.f_orb_now. the rest can be retrieved elsewhere
